<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B02%5D%20-%20Analisis_Cluster/%5B01%5D%20-%20Notebooks/E4_Profiling_de_Clusters_Mall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E4 · Perfila y nombra tus clusters - Análisis cluster

## Introducción

> Un cluster sin nombre no sirve: el valor está en entender qué hay dentro de cada grupo y
> poder accionarlo.

El clustering devuelve grupos sin significado (0, 1, 2...). El **profiling** los convierte en
perfiles que el negocio entiende y acciona ('Premium', 'Ocasional', 'En riesgo'). Lo hacemos en
4 pasos:

1. Calcula el **perfil** de cada grupo (media o mediana por variable).
2. Compáralo con la **media global** (un índice tipo 2,8x revela en qué destaca).
3. Mira el **tamaño** de cada grupo.
4. Ponle un **nombre** claro y asígnale una **acción**.

## Objetivos del ejercicio

- Calcular el **perfil medio** de cada cluster y su **índice vs la media global**.
- Identificar las **variables que más separan** los grupos.
- Entregar una **ficha de segmento**: nombre + acción de negocio por grupo.

## Descripción del dataset (Mall Customers)

Reutilizamos **Mall Customers** (200 clientes reales con edad, ingresos y spending score). Es el
dataset de compras/clientes perfecto para perfilar segmentos y traducirlos a acciones.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

### 2. Cargar, escalar y agrupar

In [ ]:
import pandas as pd

def cargar_mall():
    # Mall Customers: dataset clasico y REAL de segmentacion (200 clientes).
    # Columnas: CustomerID, Gender, Age, Annual Income (k$), Spending Score (1-100).
    urls = [
        "https://raw.githubusercontent.com/tirthajyoti/Machine-Learning-with-Python/master/Datasets/Mall_Customers.csv",
        "https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/Section%2025%20-%20Hierarchical%20Clustering/Mall_Customers.csv",
    ]
    for u in urls:
        try:
            return pd.read_csv(u).rename(columns={"Genre": "Gender"})
        except Exception:
            continue
    raise RuntimeError("No se pudo descargar Mall Customers")

In [ ]:
df = cargar_mall()
cols_cluster = ["Annual Income (k$)", "Spending Score (1-100)"]            # agrupamos por comportamiento
cols_perfil = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]       # perfilamos con todo
X_esc = StandardScaler().fit_transform(df[cols_cluster])

# Elegimos K por silhouette (como en E1) y agrupamos
mejor_k, mejor_sil = None, -1
for k in range(2, 7):
    lab = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=0).fit_predict(X_esc)
    s = silhouette_score(X_esc, lab)
    if s > mejor_sil:
        mejor_k, mejor_sil = k, s

df["cluster"] = KMeans(n_clusters=mejor_k, init="k-means++", n_init=10, random_state=0).fit_predict(X_esc)
print(f"K = {mejor_k} (silhouette {mejor_sil:.3f})")
print("Tamaño de cada grupo:")
print(df["cluster"].value_counts().sort_index())

### 3. Perfil medio de cada grupo (paso 1)

In [ ]:
perfil = df.groupby("cluster")[cols_perfil].mean().round(1)
perfil["n_clientes"] = df["cluster"].value_counts().sort_index()
perfil

### 4. Índice vs la media global (paso 2)

In [ ]:
indice = (df.groupby("cluster")[cols_perfil].mean() / df[cols_perfil].mean()).round(2)
print("Cada celda: cuántas veces la media del grupo respecto al cliente medio (1.00 = igual)")
indice

### 5. ¿Qué variables separan los grupos? (discriminantes)

Las variables útiles para el profiling son las que **más cambian** entre clusters. Las medimos
por la dispersión de las medias de cada grupo.

In [ ]:
perfil_esc = pd.DataFrame(StandardScaler().fit_transform(df[cols_perfil]), columns=cols_perfil)
perfil_esc["cluster"] = df["cluster"].values
discrimina = perfil_esc.groupby("cluster").mean().std().sort_values(ascending=False)
print("Variables ordenadas por cuánto separan los grupos (en escala estandarizada):")
print(discrimina.round(2))

### 6. La ficha del segmento (pasos 3 y 4): nombre + acción

Con el índice delante, traducimos cada grupo a un nombre y una acción. (Ajusta los nombres a
los números que te hayan salido: mira qué grupo tiene gasto alto, cuál ingresos altos, etc.)

In [ ]:
# Sugerencia automática de nombre según ingreso y spending (ajústalo si hace falta)
def nombrar(fila):
    ingreso_alto = fila["Annual Income (k$)"] >= df["Annual Income (k$)"].mean()
    gasto_alto = fila["Spending Score (1-100)"] >= df["Spending Score (1-100)"].mean()
    if ingreso_alto and gasto_alto:
        return "Premium (gasta y puede)", "Programa de fidelización y upselling"
    if ingreso_alto and not gasto_alto:
        return "Ahorrador con potencial", "Incentivos para activar gasto"
    if not ingreso_alto and gasto_alto:
        return "Entusiasta sensible al precio", "Ofertas y packs de valor"
    return "Ocasional / bajo valor", "Campaña de bajo coste o reactivación"

medias = df.groupby("cluster")[cols_perfil].mean()
ficha = pd.DataFrame(
    [nombrar(medias.loc[c]) for c in medias.index],
    columns=["nombre", "accion"], index=medias.index)
ficha["n_clientes"] = df["cluster"].value_counts().sort_index()
ficha.index.name = "cluster"
ficha

### Reflexión

1. ¿Qué grupo es el más valioso para el negocio y por qué?
2. Mirando el índice vs media global, ¿qué define a cada segmento en una frase?
3. ¿Usarías media o mediana para el perfil? ¿Cuándo cambia la decisión?
4. ¿Hay algún grupo tan pequeño que debería tratarse con cuidado (posible ruido)?